# NTD/NTAD Freight Analysis Framework Data Quality Evaluation for Year 2026

**Dataset source requested:** National Transit Database / National Transportation Atlas Database Freight Analysis Framework (FAF) dataset, map URL: <https://geodata.bts.gov/maps/b33d6a096bb94ca6b5cf5d96bf68a682>  
**Local file evaluated:** `data/csv.csv`  
**Assessment date:** 2026-06-17  

This notebook evaluates the provided CSV dataframe for schema consistency, completeness, outliers, anomalous values, stale data risk, integration issues, and readiness for production-grade analytics.

> Important scope note: the provided file does not contain a `Year` field. The only temporal/version marker is `VERSION`, and every row is `V2021.05`. Therefore, the local dataframe cannot independently prove that it represents 2026 data.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path('data/csv.csv')
EXPECTED_YEAR = 2026

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Loaded {DATA_PATH} with {df.shape[0]:,} rows and {df.shape[1]:,} columns')
df.head()

Loaded data\csv.csv with 487,394 rows and 53 columns


,OBJECTID,ID,LENGTH,DIR,DATA1,VERSION,Class,Class_Description,Road_Name,Sign_Rte,Rte_Type,Rte_Number,Rte_Qualifier,Country,STATE,STFIPS,County_Name,CTFIPS,Urban_Code,FAFZONE,Status,F_Class,Facility_Type,NHS,STRAHNET,NHFN,Truck,AB_Lanes,BA_Lanes,Speed_Limit,Toll_Type,Toll_Name,Toll_Link,Toll_Link_Name,HPMS_USA_RouteID,HPMS_Begin_Point,HPMS_End_Point,BorderState1,BorderState2,BorderFAF1,BorderFAF2,TRUCKTOLL,BorderLink,AddedBorderTime,AdjustSpeed,AdjustReason,AB_FinalSpeed,BA_FinalSpeed,AB_CombinedSpeed,BA_CombinedSpeed,AB_FreeFlowTime,BA_FreeFlowTime,Shape__Length
0,1,1,0.0541,1,1324805,V2021.05,19.0000,Facility Access/Circulator Road,PETERSBURG FERRY TERMINAL RD,NaN,NaN,NaN,NaN,USA,AK,2.0000,PETERSBURG,"2,195.0000","99,999.0000",20.0000,1.0000,5.0000,NaN,NaN,NaN,NaN,NaN,1.0000,NaN,10.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,43.0000,43.0000,43.0000,43.0000,0.0755,0.0755,0.0010
1,2,2,0.1149,0,1324806,V2021.05,19.0000,Facility Access/Circulator Road,PETERSBURG FERRY TERMINAL RD,NaN,NaN,NaN,NaN,USA,AK,2.0000,PETERSBURG,"2,195.0000","99,999.0000",20.0000,1.0000,5.0000,NaN,NaN,NaN,NaN,NaN,1.0000,1.0000,10.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,43.0000,43.0000,43.0000,43.0000,0.1603,0.1603,0.0020
2,3,3,0.0324,1,1324807,V2021.05,19.0000,Facility Access/Circulator Road,PETERSBURG FERRY TERMINAL RD,NaN,NaN,NaN,NaN,USA,AK,2.0000,PETERSBURG,"2,195.0000","99,999.0000",20.0000,1.0000,5.0000,NaN,NaN,NaN,NaN,NaN,1.0000,NaN,10.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,43.0000,43.0000,43.0000,43.0000,0.0453,0.0453,0.0006
3,4,4,0.6263,0,126521,V2021.05,14.0000,Arterial or Major Collector,NORDIC DR,NaN,NaN,NaN,NaN,USA,AK,2.0000,PETERSBURG,"2,195.0000","99,999.0000",20.0000,1.0000,4.0000,2.0000,NaN,NaN,NaN,NaN,1.0000,1.0000,35.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.0000,28.0000,28.0000,28.0000,1.3420,1.3420,0.0151
4,5,5,15.2230,0,1324808,V2021.05,41.0000,Ferry,WRANGELL-PETERSBURG FERRY,NaN,NaN,NaN,NaN,USA,AK,2.0000,PETERSBURG,"2,195.0000","99,999.0000",20.0000,1.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0000,3.0000,WRANGELL-PETERSBURG FERRY,1.0000,WRANGELL-PETERSBURG FERRY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,358.0000,NaN,NaN,NaN,NaN,10.0000,10.0000,10.0000,10.0000,91.3380,91.3380,0.3444


## 1. Dataset Overview

The file appears to be a Freight Analysis Framework network links extract. Each record represents a network segment or connector with identifiers, segment length, directional attributes, road classification, route naming, location fields, truck/toll/border flags, speeds, travel times, and geometry length.

High-level observed profile from the local CSV:

- Rows: **487,394**
- Columns: **53**
- Unique row identifiers: `OBJECTID`, `ID`, and `DATA1` are all unique and non-null.
- Full duplicate rows: **0**
- Countries represented: **USA** and **CAN**
- States/provinces represented in `STATE`: **58** distinct codes, including Canadian provinces and one invalid-looking `I` value.
- Temporal marker: **all rows are `VERSION = V2021.05`**
- Explicit 2026 indicator: **not present**

In [2]:
profile = {
    'rows': len(df),
    'columns': df.shape[1],
    'duplicate_rows': int(df.duplicated().sum()),
    'version_values': df['VERSION'].value_counts(dropna=False).to_dict() if 'VERSION' in df else {},
    'year_columns': [c for c in df.columns if 'year' in c.lower() or c.lower() in {'yr', 'report_year', 'data_year'}],
}
profile

{'rows': 487394,
 'columns': 53,
 'duplicate_rows': 0,
 'version_values': {'V2021.05': 487394},
 'year_columns': []}

## 2. Data Structure, Schema, and Metadata Assessment

The schema is wide but understandable for a transportation network-link dataset. It mixes identifiers, categorical codes, human-readable labels, geography, route descriptors, toll/border attributes, numeric speeds, travel times, and geometry length.

Important schema observations:

- No explicit `Year`, `reporting_period`, `effective_date`, `published_date`, or `last_updated` column exists in the CSV.
- `VERSION` is constant (`V2021.05`), so it cannot support comparison across reporting periods inside this file.
- Several coded fields lack embedded domain descriptions (`Status`, `F_Class`, `Facility_Type`, `NHS`, `STRAHNET`, `NHFN`, `Toll_Type`, etc.). Production analytics would require authoritative lookup tables.
- `STATE` combines U.S. state abbreviations, Canadian province/territory codes, and one anomalous code (`I`), while `STFIPS` is missing for Canadian rows. This is expected for some cross-border network data but creates integration risk if downstream systems assume U.S.-only FIPS geography.
- `Class` and `Class_Description` are not consistently normalized. Several descriptions contain spelling variants and apparent label drift for the same class code.

In [3]:
schema = pd.DataFrame({
    'column': df.columns,
    'dtype': [str(df[c].dtype) for c in df.columns],
    'non_null': [int(df[c].notna().sum()) for c in df.columns],
    'missing': [int(df[c].isna().sum()) for c in df.columns],
    'missing_pct': [df[c].isna().mean() * 100 for c in df.columns],
    'unique_values': [int(df[c].nunique(dropna=True)) for c in df.columns],
})
schema.sort_values('missing_pct', ascending=False)

,column,dtype,non_null,missing,missing_pct,unique_values
42,BorderLink,float64,168,487226,99.9655,1
43,AddedBorderTime,float64,168,487226,99.9655,1
33,Toll_Link_Name,object,1135,486259,99.7671,659
32,Toll_Link,float64,1243,486151,99.7450,1
41,TRUCKTOLL,float64,3245,484149,99.3342,2266
40,BorderFAF2,float64,4172,483222,99.1440,129
39,BorderFAF1,float64,4172,483222,99.1440,127
26,Truck,object,4365,483029,99.1044,3
38,BorderState2,object,5863,481531,98.7971,46
37,BorderState1,object,5983,481411,98.7725,47


In [4]:
# Fields with very high missingness are often sparse attributes, but they still need explicit business rules.
high_missing = schema[schema['missing_pct'] >= 50].sort_values('missing_pct', ascending=False)
high_missing

,column,dtype,non_null,missing,missing_pct,unique_values
43,AddedBorderTime,float64,168,487226,99.9655,1
42,BorderLink,float64,168,487226,99.9655,1
33,Toll_Link_Name,object,1135,486259,99.7671,659
32,Toll_Link,float64,1243,486151,99.7450,1
41,TRUCKTOLL,float64,3245,484149,99.3342,2266
39,BorderFAF1,float64,4172,483222,99.1440,127
40,BorderFAF2,float64,4172,483222,99.1440,129
26,Truck,object,4365,483029,99.1044,3
38,BorderState2,object,5863,481531,98.7971,46
37,BorderState1,object,5983,481411,98.7725,47


In [5]:
# Class code to label consistency check
class_label_counts = (
    df.groupby(['Class', 'Class_Description'], dropna=False)
      .size()
      .reset_index(name='rows')
      .sort_values(['Class', 'rows'], ascending=[True, False])
)
classes_with_multiple_labels = (
    class_label_counts.groupby('Class', dropna=False)
    .size()
    .reset_index(name='label_count')
    .query('label_count > 1')
)
class_label_counts[class_label_counts['Class'].isin(classes_with_multiple_labels['Class'])]

,Class,Class_Description,rows
4,14.0000,Arterial or Major Collector,200129
6,14.0000,Frontage/Service Road,3
3,14.0000,Arterial or Major Collecor,1
5,14.0000,Artial or Major Collector,1
13,16.0000,Frontage/Service Road,11226
15,16.0000,Frontage/Servise Road,13
9,16.0000,Collector/Distributor Lane,12
11,16.0000,Frontage/Serivce Road,12
8,16.0000,Arterial or Major Collector,6
12,16.0000,Frontage/Service Rd,2


## 3. Data Quality Findings and Identified Issues

### Finding 1: The file is not demonstrably a 2026 dataset

The assignment asks for an evaluation for **Year 2026**, but the CSV has no year field and every record has `VERSION = V2021.05`. Unless the source owner confirms that `V2021.05` remains the current authoritative release for 2026, this dataset should be treated as stale or at least temporally unverified.

**Impact:** high. Time-sensitive reporting, network-change analysis, toll modeling, and production decision support may rely on stale infrastructure, speed, routing, or toll information.

In [6]:
version_summary = df['VERSION'].value_counts(dropna=False).rename_axis('VERSION').reset_index(name='rows')
year_candidates = [c for c in df.columns if 'year' in c.lower() or 'date' in c.lower() or c.lower() in {'yr'}]
print('Year/date candidate columns:', year_candidates)
version_summary

Year/date candidate columns: []


,VERSION,rows
0,V2021.05,487394


### Finding 2: Missingness is extensive in many business-critical fields

Some sparse fields are naturally optional, such as toll or border fields. However, the dataset has very high missingness in route linkage, HPMS references, truck restrictions, network flags, and lane counts. This creates ambiguity unless null semantics are documented.

Observed examples:

- `BorderLink` and `AddedBorderTime`: **99.97% missing**
- `Toll_Link_Name`: **99.77% missing**
- `Toll_Link`: **99.74% missing**
- `TRUCKTOLL`: **99.33% missing**
- `Truck`: **99.10% missing**
- `AdjustSpeed` and `AdjustReason`: about **92% missing**
- `NHFN`: **82.18% missing**
- `HPMS_USA_RouteID`, `HPMS_Begin_Point`, `HPMS_End_Point`: about **79.6% missing**
- `STRAHNET`: **79.13% missing**
- `BA_Lanes`: **68.44% missing**
- route descriptors `Sign_Rte`, `Rte_Type`, `Rte_Number`: about **51.46% missing**

**Impact:** medium to high, depending on use case. Nulls must not be interpreted as false or zero without documented rules.

In [7]:
missing_report = schema[['column', 'missing', 'missing_pct', 'unique_values']].sort_values('missing_pct', ascending=False)
missing_report.head(30)

,column,missing,missing_pct,unique_values
42,BorderLink,487226,99.9655,1
43,AddedBorderTime,487226,99.9655,1
33,Toll_Link_Name,486259,99.7671,659
32,Toll_Link,486151,99.7450,1
41,TRUCKTOLL,484149,99.3342,2266
40,BorderFAF2,483222,99.1440,129
39,BorderFAF1,483222,99.1440,127
26,Truck,483029,99.1044,3
38,BorderState2,481531,98.7971,46
37,BorderState1,481411,98.7725,47


### Finding 3: Categorical labels contain spelling variants and inconsistent code-label mappings

`Class` should be a stable coded field, but `Class_Description` contains multiple variants for the same class code. Examples include:

- `Class = 16`: `Frontage/Service Road`, `Frontage/Servise Road`, `Frontage/Serivce Road`, `Froontage/Service Road`, `Frontage/Service Roaed`, `Frontasge/Service Road`, and one value of `Washington Ave`.
- `Class = 23`: `Collector/Distributor Lane` and `Collector/Distirbutor Lane`.
- `Class = 14`: `Arterial or Major Collector`, `Artial or Major Collector`, `Arterial or Major Collecor`, and a few rows labeled `Frontage/Service Road`.
- `Class = 19`: `Facility Access/Circulator Road`, `Facility Accss/Circulator Road`, `Facilty Access/Circulator Road`, and one row labeled `Arterial or Major Collector`.

**Impact:** medium. Reports grouped by description will fragment categories and produce misleading counts unless downstream logic groups by authoritative code or cleans descriptions.

In [8]:
# Show only class codes whose description is not one-to-one.
multi_label_classes = class_label_counts.groupby('Class', dropna=False).filter(lambda x: len(x) > 1)
multi_label_classes

,Class,Class_Description,rows
4,14.0000,Arterial or Major Collector,200129
6,14.0000,Frontage/Service Road,3
3,14.0000,Arterial or Major Collecor,1
5,14.0000,Artial or Major Collector,1
13,16.0000,Frontage/Service Road,11226
15,16.0000,Frontage/Servise Road,13
9,16.0000,Collector/Distributor Lane,12
11,16.0000,Frontage/Serivce Road,12
8,16.0000,Arterial or Major Collector,6
12,16.0000,Frontage/Service Rd,2


### Finding 4: Numeric anomalies exist in speeds, times, lanes, length, and tolls

Observed anomalies include:

- `LENGTH` has **2 zero-length records**.
- `BA_FinalSpeed` has **26 negative values** and **333 zero values**.
- `BA_CombinedSpeed` has **359 zero values**.
- `BA_FreeFlowTime` has **26 negative values**.
- `AB_Lanes` has **165 zero-lane records** and **15 records above 8 lanes**, with a maximum of **17**.
- `BA_Lanes` has **158 zero-lane records** and **2 records above 8 lanes**, with a maximum of **12**.
- `TRUCKTOLL` is highly skewed, with a maximum of **6,789**. Some high values are ferry-related and may be valid, but they require unit and vehicle-class documentation.
- `LENGTH` has a maximum of **739.86**, driven by ferry or long cross-border segments. These may be valid but should be typed and handled separately in travel-time analytics.

**Impact:** high for routing, speed, travel-time, lane-capacity, and cost analytics if these values are not validated or interpreted by facility type.

In [9]:
numeric_cols = df.select_dtypes(include='number').columns
numeric_profile = df[numeric_cols].describe(percentiles=[.001, .01, .05, .25, .5, .75, .95, .99, .999]).T
numeric_profile

,count,mean,std,min,0.1%,1%,5%,25%,50%,75%,95%,99%,99.9%,max
OBJECTID,"487,394.0000","243,697.5000","140,698.6729",1.0000,488.3930,"4,874.9300","24,370.6500","121,849.2500","243,697.5000","365,545.7500","463,024.3500","482,520.0700","486,906.6070","487,394.0000"
ID,"487,394.0000","974,811.7542","562,795.9709",1.0000,"1,832.3930","19,496.9300","97,552.6500","487,447.2500","974,863.5000","1,462,279.7500","1,852,174.3500","1,930,038.0700","1,947,672.6070","1,949,504.0000"
LENGTH,"487,394.0000",1.2068,3.5849,0.0000,0.0058,0.0143,0.0283,0.1265,0.2955,0.7816,5.9769,14.2421,30.7254,739.8586
DIR,"487,394.0000",0.6533,0.4776,-1.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
DATA1,"487,394.0000","893,214.6019","346,147.4616",2.0000,"1,374.7860","52,396.5100","225,141.6500","695,622.2500","960,042.0000","1,178,869.7500","1,327,054.3500","1,353,877.0700","1,359,471.6070","1,359,962.0000"
Class,"487,390.0000",15.9032,5.1621,11.0000,11.0000,11.0000,11.0000,12.0000,14.0000,22.0000,22.0000,23.0000,50.0000,50.0000
STFIPS,"485,636.0000",29.7736,16.1013,1.0000,1.0000,1.0000,6.0000,17.0000,30.0000,45.0000,53.0000,55.0000,56.0000,56.0000
CTFIPS,"481,840.0000","29,867.9663","16,135.1035","1,001.0000","1,013.0000","1,097.0000","6,011.0000","17,031.0000","30,021.0000","45,061.0000","53,067.0000","55,127.0000","56,037.0000","56,045.0000"
Urban_Code,"481,840.0000","66,716.2281","32,796.5676",199.0000,280.0000,"1,495.0000","5,707.0000","41,212.0000","74,179.0000","99,998.0000","99,999.0000","99,999.0000","99,999.0000","99,999.0000"
FAFZONE,"485,636.0000",301.8284,161.6095,11.0000,11.0000,19.0000,61.0000,171.0000,300.0000,459.0000,539.0000,559.0000,560.0000,560.0000


In [10]:
range_checks = []
for c in ['LENGTH','Shape__Length','Speed_Limit','AB_FinalSpeed','BA_FinalSpeed',
          'AB_CombinedSpeed','BA_CombinedSpeed','AB_FreeFlowTime','BA_FreeFlowTime',
          'AB_Lanes','BA_Lanes','TRUCKTOLL']:
    s = pd.to_numeric(df[c], errors='coerce')
    range_checks.append({
        'column': c,
        'missing': int(s.isna().sum()),
        'zero': int((s == 0).sum()),
        'negative': int((s < 0).sum()),
        'min': s.min(),
        'p01': s.quantile(.01),
        'median': s.median(),
        'p99': s.quantile(.99),
        'max': s.max(),
    })
range_checks = pd.DataFrame(range_checks)
range_checks

,column,missing,zero,negative,min,p01,median,p99,max
0,LENGTH,0,2,0,0.0000,0.0143,0.2955,14.2421,739.8586
1,Shape__Length,0,0,0,0.0000,0.0002,0.0049,0.2382,17.1758
2,Speed_Limit,232,0,0,5.0000,25.0000,45.0000,75.0000,85.0000
3,AB_FinalSpeed,3519,0,0,4.4684,22.0000,40.0000,75.0000,80.0000
4,BA_FinalSpeed,3519,333,26,-6.7500,22.0000,38.5000,75.0000,83.0000
5,AB_CombinedSpeed,3519,0,0,4.4684,25.0000,40.0000,75.0000,80.0000
6,BA_CombinedSpeed,3519,359,0,0.0000,25.0000,38.5000,75.0000,83.0000
7,AB_FreeFlowTime,3519,2,0,0.0000,0.0222,0.4461,19.0051,"4,439.1515"
8,BA_FreeFlowTime,3852,2,26,-6.4270,0.0233,0.4516,18.9929,"4,439.1515"
9,AB_Lanes,24188,165,0,0.0000,1.0000,2.0000,5.0000,17.0000


In [11]:
# Concrete examples of invalid or suspicious directional speed/time values.
speed_time_issue_mask = (
    (pd.to_numeric(df['BA_FinalSpeed'], errors='coerce') <= 0) |
    (pd.to_numeric(df['BA_CombinedSpeed'], errors='coerce') <= 0) |
    (pd.to_numeric(df['BA_FreeFlowTime'], errors='coerce') < 0)
)
example_cols = ['OBJECTID','ID','LENGTH','DIR','STATE','Road_Name','Class','Class_Description',
                'AB_Lanes','BA_Lanes','Speed_Limit','AB_FinalSpeed','BA_FinalSpeed',
                'AB_CombinedSpeed','BA_CombinedSpeed','AB_FreeFlowTime','BA_FreeFlowTime']
df.loc[speed_time_issue_mask, example_cols].head(20)

,OBJECTID,ID,LENGTH,DIR,STATE,Road_Name,Class,Class_Description,AB_Lanes,BA_Lanes,Speed_Limit,AB_FinalSpeed,BA_FinalSpeed,AB_CombinedSpeed,BA_CombinedSpeed,AB_FreeFlowTime,BA_FreeFlowTime
517,518,2054,0.2561,1,HI,LANUI CIR,19.0000,Facility Access/Circulator Road,2.0000,NaN,30.0000,27.0000,0.0000,27.0000,0.0000,0.5690,NaN
521,522,2058,0.0582,1,HI,LANUI CIR,19.0000,Facility Access/Circulator Road,2.0000,NaN,30.0000,27.0000,0.0000,27.0000,0.0000,0.1294,NaN
923,924,3612,0.0248,1,HI,KAMEHAMEHA HWY,14.0000,Arterial or Major Collector,2.0000,NaN,35.0000,31.5000,0.0000,31.5000,0.0000,0.0473,NaN
960,961,3841,0.0693,1,HI,RAMP,22.0000,Ramp,2.0000,NaN,35.0000,31.5000,0.0000,31.5000,0.0000,0.1319,NaN
974,975,3855,0.1358,1,HI,RAMP,22.0000,Ramp,3.0000,NaN,35.0000,31.5000,0.0000,31.5000,0.0000,0.2587,NaN
975,976,3856,0.0182,1,HI,RAMP,22.0000,Ramp,3.0000,NaN,35.0000,31.5000,0.0000,31.5000,0.0000,0.0348,NaN
978,979,3859,0.1826,1,HI,RAMP,22.0000,Ramp,1.0000,NaN,35.0000,31.5000,0.0000,31.5000,0.0000,0.3478,NaN
1139,1140,4404,0.1187,1,HI,OMALLEY BLVD,14.0000,Arterial or Major Collector,3.0000,NaN,35.0000,31.5000,0.0000,31.5000,0.0000,0.2261,NaN
1145,1146,4410,0.0304,1,HI,RAMP,22.0000,Ramp,2.0000,NaN,35.0000,31.5000,0.0000,31.5000,0.0000,0.0579,NaN
1146,1147,4411,0.0482,1,HI,RAMP,22.0000,Ramp,1.0000,NaN,35.0000,31.5000,0.0000,31.5000,0.0000,0.0919,NaN


In [12]:
# Travel time consistency check: expected time in minutes = LENGTH / speed * 60.
time_consistency = []
for direction in ['AB', 'BA']:
    length = pd.to_numeric(df['LENGTH'], errors='coerce')
    speed = pd.to_numeric(df[f'{direction}_CombinedSpeed'], errors='coerce')
    observed = pd.to_numeric(df[f'{direction}_FreeFlowTime'], errors='coerce')
    expected = length / speed * 60
    valid = length.notna() & speed.gt(0) & observed.notna()
    diff = (observed - expected).abs()
    time_consistency.append({
        'direction': direction,
        'valid_rows': int(valid.sum()),
        'rows_diff_gt_0_01_min': int((valid & diff.gt(0.01)).sum()),
        'rows_diff_gt_1_min': int((valid & diff.gt(1)).sum()),
        'max_abs_diff_minutes': diff[valid].max(),
    })
pd.DataFrame(time_consistency)

,direction,valid_rows,rows_diff_gt_0_01_min,rows_diff_gt_1_min,max_abs_diff_minutes
0,AB,483875,34176,1605,69.1886
1,BA,483516,34459,1623,69.1886


### Finding 5: Directional fields are structurally difficult to interpret

`DIR` has three values: `1`, `0`, and `-1`. `BA_Lanes` is missing for most one-way or directional records, while `AB_Lanes` is usually populated. This may be intentional, but the schema does not encode enough null semantics to support automated interpretation.

The pattern suggests that `BA_*` fields may be inapplicable for many `DIR = 1` records, but the file also contains zero and negative backward-direction speed/time values. Those should be normalized to null when inapplicable or corrected when the backward direction exists.

**Impact:** high for bidirectional routing and travel-time calculations.

In [13]:
pd.crosstab(
    df['DIR'],
    [df['AB_Lanes'].isna().rename('AB_Lanes_missing'), df['BA_Lanes'].isna().rename('BA_Lanes_missing')],
    dropna=False,
)

AB_Lanes_missing   False         True        
BA_Lanes_missing   False   True  False  True 
DIR                                          
-1                   110     234     0     47
 0                152972     178    29  15036
 1                   684  309028     7   9069

### Finding 6: Integration and geography risks are material

The dataset includes U.S. and Canadian rows in a single `STATE` field. Canadian records have missing `STFIPS`, `County_Name`, `CTFIPS`, `Urban_Code`, and `FAFZONE` values. This is understandable for cross-border network modeling, but it becomes a data silo/integration challenge when joining to U.S.-only geography reference tables.

Additional integration issues:

- `HPMS_USA_RouteID` and milepost fields are missing for roughly 80% of rows, limiting integration with HPMS-based datasets.
- Toll and border fields are very sparse and require separate lookup/domain metadata.
- Route names and route numbers are incomplete, which can reduce usefulness for user-facing reporting.
- `Shape__Length` exists, but the CSV does not include geometry coordinates. Geospatial validation, topology checks, and map rendering require the source geodatabase or feature service, not only this CSV.

**Impact:** medium to high. The CSV alone is insufficient for production geospatial analytics that require authoritative topology and stable crosswalks.

In [14]:
country_state = df.groupby(['Country', 'STATE'], dropna=False).size().reset_index(name='rows').sort_values('rows', ascending=False)
country_state.head(80)

,Country,STATE,rows
50,USA,TX,52919
10,USA,CA,48095
41,USA,NY,24006
29,USA,MI,19802
45,USA,PA,17015
55,USA,WI,16641
15,USA,FL,16352
42,USA,OH,16327
21,USA,IL,15389
34,USA,NC,14356


In [15]:
# Non-USA geography completeness
geo_cols = ['Country','STATE','STFIPS','County_Name','CTFIPS','Urban_Code','FAFZONE']
non_usa_geo_missing = df[df['Country'].ne('USA')][geo_cols].isna().mean().mul(100).rename('missing_pct').reset_index()
non_usa_geo_missing.rename(columns={'index': 'column'})

,column,missing_pct
0,Country,0.0000
1,STATE,0.0000
2,STFIPS,100.0000
3,County_Name,100.0000
4,CTFIPS,100.0000
5,Urban_Code,100.0000
6,FAFZONE,100.0000


### Finding 7: Schema drift across reporting periods cannot be evaluated from this single file

The file contains only one `VERSION` value and no reporting-period columns. Therefore, direct schema drift analysis across 2021, 2022, 2023, 2024, 2025, and 2026 cannot be performed from `data/csv.csv` alone.

For production governance, schema drift should be evaluated by comparing multiple published extracts or by querying the authoritative feature service metadata over time. Required checks include added/removed columns, dtype changes, domain-code changes, nullable-status changes, and changes in coded-value lookups.

In [16]:
# A compact machine-readable issue register produced from the local profile.
issue_register = pd.DataFrame([
    {'severity': 'High', 'area': 'Temporal validity', 'issue': 'No Year/date field and all rows are VERSION V2021.05', 'evidence': '487,394 rows have VERSION V2021.05; no year/date candidate columns'},
    {'severity': 'High', 'area': 'Directional speed/time', 'issue': 'Invalid BA speed/time values', 'evidence': 'BA_FinalSpeed: 26 negative and 333 zero; BA_CombinedSpeed: 359 zero; BA_FreeFlowTime: 26 negative'},
    {'severity': 'High', 'area': 'Completeness', 'issue': 'Sparse route, HPMS, truck, toll, border, and lane attributes', 'evidence': 'BA_Lanes 68.44% missing; HPMS route fields about 79.6% missing; Truck 99.10% missing'},
    {'severity': 'Medium', 'area': 'Categorical consistency', 'issue': 'Class descriptions are not normalized', 'evidence': 'Multiple spelling variants and inconsistent labels for Class 14, 16, 19, 23, and 33'},
    {'severity': 'Medium', 'area': 'Geographic integration', 'issue': 'U.S. and Canadian geography mixed in one schema', 'evidence': 'Country includes USA and CAN; Canadian rows lack U.S. FIPS/county fields'},
    {'severity': 'Medium', 'area': 'Outliers', 'issue': 'Very long links and high truck toll values require facility-specific interpretation', 'evidence': 'LENGTH max 739.86; TRUCKTOLL max 6,789'},
    {'severity': 'Medium', 'area': 'Schema drift', 'issue': 'Cannot evaluate drift across periods from one versioned file', 'evidence': 'Only one VERSION value is present'},
    {'severity': 'Low', 'area': 'Uniqueness', 'issue': 'No duplicate row or duplicate primary identifiers found', 'evidence': 'OBJECTID, ID, and DATA1 are unique and non-null'},
])
issue_register

,severity,area,issue,evidence
0,High,Temporal validity,No Year/date field and all rows are VERSION V2...,"487,394 rows have VERSION V2021.05; no year/da..."
1,High,Directional speed/time,Invalid BA speed/time values,BA_FinalSpeed: 26 negative and 333 zero; BA_Co...
2,High,Completeness,"Sparse route, HPMS, truck, toll, border, and l...",BA_Lanes 68.44% missing; HPMS route fields abo...
3,Medium,Categorical consistency,Class descriptions are not normalized,Multiple spelling variants and inconsistent la...
4,Medium,Geographic integration,U.S. and Canadian geography mixed in one schema,Country includes USA and CAN; Canadian rows la...
5,Medium,Outliers,Very long links and high truck toll values req...,"LENGTH max 739.86; TRUCKTOLL max 6,789"
6,Medium,Schema drift,Cannot evaluate drift across periods from one ...,Only one VERSION value is present
7,Low,Uniqueness,No duplicate row or duplicate primary identifi...,"OBJECTID, ID, and DATA1 are unique and non-null"


## 4. Risk Assessment Related to Data Reliability and Usability

| Risk area | Risk level | Assessment |
|---|---:|---|
| 2026 temporal validity | High | The local extract is versioned `V2021.05` and lacks a year/effective-date field. It cannot be accepted as current 2026 data without source-owner confirmation or a refreshed extract. |
| Production routing/travel-time analytics | High | Negative/zero backward speeds and negative travel times can break route-cost calculations or understate travel impedance. Directional null semantics are not explicit. |
| Reporting and KPI aggregation | Medium-High | Inconsistent category descriptions and sparse route/geography fields can fragment reports and create misleading groupings. |
| Data integration | Medium-High | Missing HPMS identifiers, mixed U.S./Canadian geography, and absent geometry coordinates limit joins and geospatial validation from the CSV alone. |
| Toll/cost analytics | Medium | Toll fields are sparse and highly skewed. Ferry tolls may be valid, but units, vehicle classes, and effective dates are not documented in the file. |
| Schema governance | Medium | Only one version is present, so schema drift cannot be measured from the provided file. |
| Identifier integrity | Low | Primary identifiers are unique and non-null, and there are no exact duplicate records. |

Overall reliability risk is **High** for a production-grade analytics platform if this CSV is used as-is for 2026. It is more suitable as an exploratory or reference extract pending validation against the authoritative source.

## 5. Recommendations for Cleansing, Governance, and Validation

1. **Resolve temporal authority before use.** Obtain a current 2026 export or authoritative metadata proving that `V2021.05` is still the valid release. Add fields such as `source_release`, `effective_date`, `published_date`, `retrieved_at`, and `data_year`.
2. **Create a formal data dictionary.** Document each code field, units, null semantics, applicable facility types, and valid value domains.
3. **Normalize categorical labels.** Treat coded fields such as `Class` as authoritative and map `Class_Description` through a controlled lookup table. Correct spelling variants and reject unexpected labels.
4. **Validate directional metrics.** Enforce rules for `DIR`, `AB_*`, and `BA_*` fields. Convert inapplicable backward-direction values to null, and reject negative speeds or negative travel times.
5. **Separate optional sparse domains.** Move toll, border, truck restriction, STRAHNET, NHFN, and HPMS details into documented auxiliary tables or explicitly flag them as optional attributes.
6. **Add geospatial validation.** Use the source feature service/geodatabase for geometry topology checks, link connectivity, CRS validation, and geometry-length consistency. The CSV alone is not enough for spatial production validation.
7. **Implement automated quality gates.** At ingestion, test uniqueness, required fields, valid domains, numeric ranges, speed/time consistency, route/geography join rates, and row-count/schema drift relative to prior releases.
8. **Track stale elements.** Toll rates, truck restrictions, speed assumptions, network status, and border-delay assumptions should have effective dates and refresh ownership.
9. **Define integration crosswalks.** Maintain explicit crosswalks for FAF zones, HPMS IDs, county/FIPS, Canadian province geography, route identifiers, and facility classes.
10. **Publish exception reports.** Keep a recurring issue register for anomalies that are valid by business rule, such as ferries and cross-border links, so they are not repeatedly flagged without context.

## 6. Production Readiness Conclusion

Based on the provided `data/csv.csv`, the dataset is **not sufficiently reliable as-is to support a production-grade analytics platform for Year 2026**.

The main blocker is temporal validity: the file lacks any explicit 2026 marker and all rows are marked `V2021.05`. Even if the dataset is otherwise structurally useful, production use for 2026 requires confirmation that this is the current authoritative release or replacement with a current extract.

Secondary blockers include high missingness in integration fields, inconsistent categorical descriptions, directional speed/time anomalies, sparse and undocumented toll/border/truck attributes, and the lack of geometry coordinates in the CSV. These issues do not make the dataset unusable, but they require controlled ingestion, cleansing rules, metadata enrichment, and validation against authoritative source systems before production analytics, reporting, or decision-making.

Recommended decision: **conditionally reject for production 2026 use until refreshed/source-validated and governed through automated quality checks.** After temporal validation and cleansing, the dataset could support a production platform as a curated network reference layer rather than as a raw standalone CSV.

In [ ]:
# Optional: save compact tables to variables for downstream export if desired.
outputs = {
    'schema': schema,
    'missing_report': missing_report,
    'range_checks': range_checks,
    'issue_register': issue_register,
}
print('Notebook analysis objects prepared:', ', '.join(outputs.keys()))